In [2]:
import os
import sys

import numpy as np

import flashlfq_py

PROTON_MASS = 1.007276466879  # mzLib Chemistry/Constants.cs

REPO = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
TESTDATA = os.path.join(REPO, "mzLib", "Test", "FlashLFQ", "TestData")
PSMTSV = os.path.join(TESTDATA, "AllPSMs.psmtsv")


In [3]:
def read_first_id_for_available_file():
    """Return (bare_file_name, mzml_path, mono_mass, charge, rt) for the first psmtsv row
    whose spectra file exists in TestData as an mzML."""
    with open(PSMTSV, "r", encoding="utf-8") as fh:
        header = fh.readline().rstrip("\n").split("\t")
        col = {name: i for i, name in enumerate(header)}
        fn_i = col["File Name"]
        mono_i = col["Peptide Monoisotopic Mass"]
        charge_i = col["Precursor Charge"]
        rt_i = col["Scan Retention Time"]
        full_i = col["Full Sequence"]
        for line in fh:
            cells = line.rstrip("\n").split("\t")
            bare = os.path.splitext(os.path.basename(cells[fn_i].strip().strip('"')))[0]
            mzml = os.path.join(TESTDATA, bare + ".mzML")
            if not os.path.exists(mzml):
                continue
            mono_token = cells[mono_i].strip().strip('"').split("|")[0]
            try:
                mono = float(mono_token)
                charge = int(float(cells[charge_i].strip().strip('"')))
                rt = float(cells[rt_i].strip().strip('"'))
            except ValueError:
                continue
            return bare, mzml, mono, charge, rt, cells[full_i].strip().strip('"')
    raise RuntimeError("no psmtsv row referenced an available mzML in TestData")